# Reproducing Timme et al. (2016) from CRCNS ssc-3

**Paper.** Timme NM, Ito S, Myroshnychenko M, Nigam S, Shimono M, Yeh F-C, Hottowy P, Litke AM,
Beggs JM (2016). High-degree neurons feed cortical computations. *PLoS Comput Biol* 12(5): e1004858.
[doi:10.1371/journal.pcbi.1004858](https://doi.org/10.1371/journal.pcbi.1004858). The claims file
was written from the version of record (PMC4861348, Europe PMC full text). Its displayed
equations, missing from that flattened text, were read from the version of record's JATS XML.
The TE procedure the paper reuses is that of its reference [44], Timme et al. (2014), PLoS One 9:
e115764 (PMC4275261).

**Data set.** [CRCNS ssc-3](https://doi.org/10.6080/K07D2S2F), `ssc-3/live/data.tar.gz` in the public
CRCNS bucket, streamed and decompressed in memory. CRCNS data sets are not versioned, so the version
is recorded as `unversioned`; the paper cites the data set by its DOI and no version, and this is
the only copy the bucket holds (`ssc-3/live/`, no `old/`). The MD5 sum of the archive, from the data
set's `checksums.md5`, is `1a5bd3d304c5442290a7c9c1f1b5def6`; the loading cell checks the streamed
bytes against it. The archive holds 25 MATLAB files, one per recording.

**Claims.**

| Claim | Tested here |
|---|---|
| data-recordings | tested |
| fig4ac-te-lognormal | tested |
| fig4bd-heavy-tailed-degree | tested |
| fig4bd-out-tail-longer | tested |
| fig5a-in-degree-uncorrelated | tested |
| fig5b-out-degree-positive | tested |
| fig5ab-correlations-beyond-null | tested |
| fig5cd-pooled-trends | tested |
| fig6-information-gain-not-rising | tested, with sampled receivers and combinations |
| table1-pid-examples | tested (no recorded data) |
| fig7b-connectivity-patterns | tested (model simulation) |
| fig7c-degree-modified-rule-matches | tested (model simulation) |
| fig7d-degree-rule-highest-conditioned-mi | tested (model simulation) |
| figs1-larger-model | tested (model simulation) |

How the TE networks are built (bins, past states, jitter, sub-networks) and where each choice comes
from is in the docstring of `hdn.py`. The significance test uses the paper's 5000 jittered
surrogates per pair, with one approximation needed to fit the time limit: for pairs whose jitter
moves more than 50 relevant sender spikes, the jittered counts are drawn from a normal distribution
with their exact moments, in place of jittering every spike. The first cell of the TE networks section compares the
two on two recordings at both time scales. A surrogate whose TE equals the real TE counts as exceeding it, and
the literal strict rule is shown as a variant; the TE networks section gives the reasons.

In [ ]:
import itertools
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit

import hdn

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
started = time.time()
recs, md5 = hdn.load_all()
print(f"CRCNS {hdn.DATASET} ({hdn.DATASET_DOI}), version {hdn.DATASET_VERSION}: {len(recs)} recordings "
      f"streamed in {time.time() - started:.0f} s")
print("MD5 of the streamed archive:", md5, "| checksums.md5:", hdn.DATA_MD5, "| equal:", md5 == hdn.DATA_MD5)

## Data check

In [ ]:
rows = []
for r in recs:
    n = np.array([len(s) for s in r["spikes"]])
    allsp = np.concatenate(r["spikes"])
    rows.append({
        "file": r["name"], "recording_id": r["recording_id"], "nNeurons": r["n_neurons_field"],
        "trains": len(r["spikes"]), "length_ms": r["length_ms"], "first_ms": allsp.min(), "last_ms": allsp.max(),
        "spikes_min": n.min(), "spikes_max": n.max(), "under_100": int((n < 100).sum()),
        "sorted": all(np.all(np.diff(s) >= 0) for s in r["spikes"]),
        "on_50us_grid": bool(np.allclose(np.round(allsp * 20) / 20, allsp)),
        "x_um": f"{np.nanmin(r['x']):.0f} to {np.nanmax(r['x']):.0f}", "y_um": f"{np.nanmin(r['y']):.0f} to {np.nanmax(r['y']):.0f}",
        "timescale": r["timescale"], "sampling": r["sampling"]})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nnNeurons equals the number of spike trains in every file:", bool((info.nNeurons == info.trains).all()))
print("Spike times sorted within every train:", bool(info.sorted.all()), "| on the 0.05 ms grid (20 kHz):", bool(info.on_50us_grid.all()))
print("Every spike within [0, recordinglength]:", bool(((info.first_ms >= 0) & (info.last_ms <= info.length_ms)).all()))
print("Recording lengths (ms):", info.length_ms.value_counts().to_dict())
print("Recording dates:", sorted(set(r[:10] for r in info.recording_id)))
# recordings deposited twice: identical spike trains in two files
sig = {}
for r in recs:
    for k, s in enumerate(r["spikes"]):
        sig.setdefault((len(s), round(float(s[0]), 2), round(float(s[-1]), 2)), []).append((r["name"], k))
shared = [v for v in sig.values() if len({f for f, _ in v}) > 1]
print("Spike trains with the same count, first and last spike in two files:", len(shared))
print("Distinct recording IDs:", info.recording_id.nunique(), "of", len(info))

The descriptor says the data are in ms at 20 kHz, with recordings of "approximately 1 hour"
(3600000 ms "for most data sets") and positions in um with the origin at the array's centre on a
2 mm by 1 mm array. The values agree: spike times sit on the 0.05 ms grid, positions span about
-1100 to 1100 um in x and -600 to 600 um in y, 24 recordings last 3600000 ms and one (DataSet19)
2887498 ms (48 min). The first ten recordings have no spike in their first 8 s. No spike train
appears in two files and the 25 recording IDs are distinct, so no recording is deposited twice.
The files hold spike times only: there are no trials or other data streams whose clock could
differ from the spikes', so the clock check of the protocol does not apply.

## data-recordings

> The analysis used 25 recordings of somatosensory cortex cultures, with 7735 neurons in total after
> removing neurons with fewer than 100 spikes in the 60 min recording, an average of 309.4 neurons
> per recording and an average firing rate of 2.10 Hz.

In [ ]:
counts = np.array([len(r["spikes"]) for r in recs])
spk = [np.array([len(s) for s in r["spikes"]]) for r in recs]
keep = [n >= 100 for n in spk]
rate_len = np.concatenate([n / (r["length_ms"] / 1000) for n, r in zip(spk, recs)])
rate_60 = np.concatenate([n / 3600.0 for n in spk])
print("Recordings:", len(recs))
print("Neurons as deposited:", counts.sum(), "| mean per recording:", round(counts.mean(), 1))
print("Neurons with 100 or more spikes:", sum(k.sum() for k in keep), "(the deposit already applies the threshold)")
print(f"Mean rate per neuron, (a) spikes over each recording's length: {rate_len.mean():.3f} Hz; "
      f"(b) spikes over 60 min: {rate_60.mean():.3f} Hz")
print(f"Rates range from {rate_len.min():.3f} to {rate_len.max():.1f} Hz")

**Result: reproduced.** The data set holds 25 recordings and 7735 neurons (309.4 per recording), every one with at least 100 spikes, so the deposit already applies the paper's threshold. The mean rate is 2.103 Hz with spikes counted over 60 min (reading b) and 2.132 Hz over each recording's own length (reading a); both are within 0.21 Hz of 2.10 Hz.

## TE networks

TE from every neuron to every other, at both time scales, with the jitter test (`hdn.te_network`).
Two details of the test are open, and every reading is run:

- Which two bins form the past state ([44]'s text gives bins t-1 and t-2, its Table 1 and the
  paper's delay windows imply t-2 and t-3; see `hdn.py`). Both are run.
- How a surrogate whose TE equals the real TE is counted. [44] takes as the p-value "the
  percentage of the TE values from jittered data that were larger than the TE value obtained from
  the data" (Transfer Entropy Analysis, paragraph 6), and the claims file (M-SIG) says "larger".
  When the jitter cannot change the counts of a pair, every surrogate ties with the real TE, so
  under a strict "larger" the pair is called significant with no evidence. Both rules are run:
  ties counted as not exceeding (strict, the literal text) and ties counted as exceeding (the
  standard permutation p-value). Each test runs to completion for both decisions.

The first cell checks the normal approximation against the full jitter of every spike on two
recordings at both bin sizes, pair by pair, under both tie rules.

In [ ]:
check = []
for name in ["DataSet1", "DataSet7"]:
    r = next(x for x in recs if x["name"] == name)
    for scale, bin_ms in hdn.TIME_SCALES.items():
        trains, T = hdn.bin_trains(r["spikes"], bin_ms, r["length_ms"])
        off = ~np.eye(len(trains), dtype=bool)
        t = time.time()
        ex = hdn.te_network(trains, T, hdn.N_SURROGATES, hdn.MAX_EXCEED, 1, 10**9, 2, 3, False)
        t_exact = time.time() - t
        ap = hdn.te_network(trains, T, hdn.N_SURROGATES, hdn.MAX_EXCEED, 1, 50, 2, 3, False)
        for rule, col in [("ties exceed", 4), ("strict", 3)]:
            e, a = (ex[col] < hdn.MAX_EXCEED) & off, (ap[col] < hdn.MAX_EXCEED) & off
            check.append({"recording": name, "scale": scale, "rule": rule, "pairs": int(off.sum()),
                          "significant_exact": int(e.sum()), "significant_approx": int(a.sum()), "both": int((e & a).sum()),
                          "exact_only": int((e & ~a).sum()), "approx_only": int((~e & a).sum()),
                          "agreement_of_union": round((e & a).sum() / max((e | a).sum(), 1), 3), "seconds_exact": round(t_exact)})
print(pd.DataFrame(check).to_string(index=False))

In [ ]:
NETS = {}
rows = []
t_all = time.time()
for past_name, (p0, p1) in hdn.PAST_READINGS.items():
    for scale, bin_ms in hdn.TIME_SCALES.items():
        for r in recs:
            trains, T = hdn.bin_trains(r["spikes"], bin_ms, r["length_ms"])
            te, nte, ran, gt, ge, n_rel, n_mov = hdn.te_network(trains, T, hdn.N_SURROGATES, hdn.MAX_EXCEED, 1, 50, p0, p1, False)
            n = len(trains)
            off = ~np.eye(n, dtype=bool)
            sig_s, sig_t = (gt < hdn.MAX_EXCEED) & off, (ge < hdn.MAX_EXCEED) & off
            NETS[past_name, scale, r["name"]] = {"trains": trains, "T": T, "te": te, "nte": nte,
                                                 "sig": {"ties": sig_t, "strict": sig_s}}
            rows.append({"past": past_name, "scale": scale, "recording": r["name"], "neurons": n,
                         "expected_by_chance": round(0.001 * n * (n - 1)),
                         "sig_ties": int(sig_t.sum()), "sig_strict": int(sig_s.sum()),
                         "strict_zero_exceed": int((sig_s & (ran == hdn.N_SURROGATES) & (gt == 0)).sum()),
                         "strict_no_movable_spike": int((sig_s & (n_mov == 0)).sum()),
                         "strict_no_relevant_spike": int((sig_s & (n_rel == 0)).sum()),
                         "density_ties": sig_t.sum() / (n * (n - 1)), "density_strict": sig_s.sum() / (n * (n - 1)),
                         "eff_N_ties": int((sig_t.any(0) | sig_t.any(1)).sum()), "eff_N_strict": int((sig_s.any(0) | sig_s.any(1)).sum())})
net_info = pd.DataFrame(rows)
net_info["eff_k_ties"] = net_info.sig_ties / net_info.eff_N_ties.clip(lower=1)
net_info["eff_k_strict"] = net_info.sig_strict / net_info.eff_N_strict.clip(lower=1)
print(f"{time.time() - t_all:.0f} s")
print(net_info.round(4).to_string(index=False))
agg = net_info.groupby(["past", "scale"]).agg(
    neurons=("neurons", "sum"), chance=("expected_by_chance", "sum"), sig_ties=("sig_ties", "sum"), sig_strict=("sig_strict", "sum"),
    strict_zero_exceed=("strict_zero_exceed", "sum"), strict_no_movable_spike=("strict_no_movable_spike", "sum"),
    strict_no_relevant_spike=("strict_no_relevant_spike", "sum"),
    density_ties_median=("density_ties", "median"), density_strict_median=("density_strict", "median"),
    eff_N_ties_mean=("eff_N_ties", "mean"), eff_N_strict_mean=("eff_N_strict", "mean"),
    eff_k_ties_mean=("eff_k_ties", "mean"), eff_k_strict_mean=("eff_k_strict", "mean"))
agg["ties_x_chance"] = agg.sig_ties / agg.chance
agg["strict_x_chance"] = agg.sig_strict / agg.chance
print()
print(agg.round(4).T.to_string())

The false-positive rate of each rule, measured on the data. Each neuron's binned train is shifted
circularly by its own random offset (10% to 90% of the recording), which keeps every neuron's rate
and bursts and removes the timing between neurons, so every pair is independent and a test at
p < 0.001 should call at most 0.1% of the pairs significant. Five recordings (every fifth), both
time scales, bins t-2 and t-3.

In [ ]:
rng = np.random.default_rng(31)
rows = []
for name in ["DataSet1", "DataSet6", "DataSet11", "DataSet16", "DataSet21"]:
    r = next(x for x in recs if x["name"] == name)
    for scale, bin_ms in hdn.TIME_SCALES.items():
        trains, T = hdn.bin_trains(r["spikes"], bin_ms, r["length_ms"])
        shifted = type(trains)()
        for tr in trains:
            shifted.append(np.sort((tr + rng.integers(int(0.1 * T), int(0.9 * T))) % T))
        out = hdn.te_network(shifted, T, hdn.N_SURROGATES, hdn.MAX_EXCEED, 2, 50, 2, 3, False)
        off = ~np.eye(len(trains), dtype=bool)
        rows.append({"recording": name, "scale": scale, "pairs": int(off.sum()),
                     "fp_rate_ties": ((out[4] < hdn.MAX_EXCEED) & off).sum() / off.sum(),
                     "fp_rate_strict": ((out[3] < hdn.MAX_EXCEED) & off).sum() / off.sum()})
fp = pd.DataFrame(rows)
print(fp.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nPooled false-positive rate, ties exceeding:", round(fp.fp_rate_ties.mul(fp.pairs).sum() / fp.pairs.sum(), 5),
      "| strict:", round(fp.fp_rate_strict.mul(fp.pairs).sum() / fp.pairs.sum(), 5), "| nominal: 0.001")

In [ ]:
# degree against firing rate (bins t-2 and t-3), Spearman per recording and time scale
rows = []
for scale in hdn.TIME_SCALES:
    for r in recs:
        rate = np.array([len(s) for s in r["spikes"]]) / (r["length_ms"] / 1000)
        for rule in ["ties", "strict"]:
            sig = NETS["t-2,t-3", scale, r["name"]]["sig"][rule]
            rows.append({"scale": scale, "recording": r["name"], "rule": rule,
                         "rho_total": stats.spearmanr(rate, sig.sum(0) + sig.sum(1))[0],
                         "rho_in": stats.spearmanr(rate, sig.sum(0))[0], "rho_out": stats.spearmanr(rate, sig.sum(1))[0]})
rate_deg = pd.DataFrame(rows)
print("Spearman correlation of degree with firing rate, median (min to max) over the 25 recordings:")
for (scale, rule), sub in rate_deg.groupby(["scale", "rule"]):
    print(f"  {scale}, {rule}: " + "; ".join(f"{c[4:]} {sub[c].median():.2f} ({sub[c].min():.2f} to {sub[c].max():.2f})"
                                          for c in ["rho_total", "rho_in", "rho_out"]))

**Which tie rule is used.** The paper states its threshold as p < 0.001 (Methods, Effective
Connectivity Detection, paragraph 5), so the test it describes calls at most 0.1% of independent
pairs significant. The false-positive rate is printed above for both rules on shifted trains; the
rule whose rate is near the stated 0.001 is the procedure the paper describes, and the other is
not (protocol: a procedure whose chance level is far from the paper's differs from it). The
analysis below uses ties counted as exceeding (`ties`), with bins t-2 and t-3, and shows the
strict rule and bins t-1 and t-2 as variants. [44] reports effective N of 241 +- 103 and
effective k of 22 +- 30 for its cortical networks over all ten of its time scales (Fig 12C and D
legend); the table above gives both for each rule here.

In [ ]:
# name: (past bins, tie rule, sub-network mode, information gain, sub-networks, combinations per n);
# the variants use 100 sub-networks per recording and 20 combinations per n, for time
CONFIGS = {
    "main": ("t-2,t-3", "ties", "strongest", True, hdn.N_SUBNETS, hdn.N_GAIN_COMBOS),
    "strict": ("t-2,t-3", "strict", "strongest", True, 100, 20),
    "past t-1,t-2": ("t-1,t-2", "ties", "strongest", True, 100, 20),
    "sub random": ("t-2,t-3", "ties", "random", False, 100, 20),
}
RES = {c: {} for c in CONFIGS}
t_all = time.time()
for c, (past_name, rule, mode, do_gain, n_sub, n_comb) in CONFIGS.items():
    for scale in hdn.TIME_SCALES:
        for r in recs:
            net = NETS[past_name, scale, r["name"]]
            seed = 1000 * list(hdn.TIME_SCALES).index(scale) + int(r["name"].replace("DataSet", ""))
            RES[c][scale, r["name"]] = hdn.analyse_recording(net["trains"], net["T"], net["sig"][rule], net["nte"], seed,
                                                             past=hdn.PAST_READINGS[past_name], sub_mode=mode, gains=do_gain,
                                                             n_sub=n_sub, gain_combos=n_comb)
    print(f"{c}: done at {time.time() - t_all:.0f} s")
rows = []
for c in CONFIGS:
    for (scale, name), a in RES[c].items():
        e = np.array(a["sub_info"]["edges"])
        rows.append({"config": c, "scale": scale, "recording": name, "subnetworks": len(a["subs"]), "draws": a["sub_info"]["draws"],
                     "draws_unconnectable": a["sub_info"]["unconnectable"], "draws_under_75": a["sub_info"]["too_few"],
                     "edges_before_trim_median": np.median(e) if len(e) else np.nan, "edges_before_trim_min": e.min() if len(e) else np.nan,
                     "subnet_triples": len(a["sub_triples"]), "full_triples_total": a["n_full_total"], "full_triples_used": len(a["full_triples"])})
summ = pd.DataFrame(rows)
print(summ[summ.config == "main"].to_string(index=False))
print()
print(summ.groupby(["config", "scale"]).agg(recordings_complete=("subnetworks", lambda v: int((v == v.max()).sum())), subnetworks=("subnetworks", "max"),
      draws=("draws", "sum"), draws_under_75=("draws_under_75", "sum"), draws_unconnectable=("draws_unconnectable", "sum"),
      edges_before_trim_median=("edges_before_trim_median", "median"), edges_before_trim_min=("edges_before_trim_min", "min")).to_string())
VARIANTS = {   # name: (config, PID form)
    "main": ("main", "claims"), "strict ties": ("strict", "claims"), "past t-1,t-2": ("past t-1,t-2", "claims"),
    "sub (b) random": ("sub random", "claims"), "PID Eq 8": ("main", "eq8")}

**Networks.** With bins t-2 and t-3, ties counted as exceeding give 55190 significant connections at
1.6 to 6.4 ms and 130029 at 3.5 to 14 ms, 19.9 and 46.9 times the 2774 expected by chance at
p < 0.001; the strict rule gives 261640 and 263599 (94 and 95 times). Of the strict rule's
connections, 247754 and 239189 had no surrogate above the real TE in all 5000, and 167255 and
109017 had no sender spike whose jitter could change the counts, so every surrogate tied. On
shifted trains, where every pair is independent, the strict rule calls 10.8% of pairs significant
(2.6% to 16.1% by recording and time scale) and the tie-counting rule 0.32% (0.04% to 1.3%),
against the nominal 0.1%. The strict rule's false-positive rate is about 100 times the stated
p < 0.001, so it is not the paper's test; the tie-counting rule's pooled rate is about three times
the nominal one, and it is used below. Its networks have an effective N of 245 and 277 and an
effective k of 8.8 and 19.5 (mean over recordings), against [44]'s 241 +- 103 and 22 +- 30 for
cortical networks; the strict rule's have 308 and 29 to 32. Degree rises with firing rate under the
tie-counting rule (median Spearman 0.75 and 0.81 for total degree), as [44] reports for its
networks; under the strict rule it does not (median -0.31 and 0.22, ranging from -0.87 to 0.93).
The normal approximation agrees less well with the full jitter under the tie-counting rule than
under the strict one: the two share 83% to 89% of the connections either finds (95.7% to 99.2%
under the strict rule), and the approximation finds 0.1% to 8.2% more.

**Sub-networks.** Draws with fewer than 75 connections among the 50 neurons are redrawn, as the
claims file says. At 3.5 to 14 ms every recording gives its 500 sub-networks. At 1.6 to 6.4 ms 21
of 25 do; within 5000 draws DataSet2 gives 6, DataSet5 none, DataSet10 380 and DataSet17 148, and
these partial sets are used as they are (DataSet5 drops out of reading (s) at that scale).

## fig4ac-te-lognormal

> The significant TE connection weights, raw and normalized, are roughly log-normally distributed at
> both time scales and vary over several orders of magnitude.

Reading (f) uses every significant connection; reading (s) the connections kept in the 500
sub-networks, each counted once per recording. The full table is for the main analysis (ties
counted as exceeding, bins t-2 and t-3, the 75 strongest connections per sub-network); the
verdict under each variant follows it.

In [ ]:
def lognorm_pmf(x, a, mu, s):
    # paper, Eq 21: alpha / (sigma sqrt(2 pi)) exp(-(log10 x - mu)^2 / (2 sigma^2))
    return a / (s * np.sqrt(2 * np.pi)) * np.exp(-(np.log10(x) - mu) ** 2 / (2 * s ** 2))

def fit_r2(x, y):
    lx = np.log10(x)
    m = (y * lx).sum() / y.sum()
    sd = np.sqrt((y * (lx - m) ** 2).sum() / y.sum())
    try:
        # start from the moments of the binned distribution; alpha starts at the log bin width
        p, _ = curve_fit(lognorm_pmf, x, y, p0=[lx[1] - lx[0], m, sd], maxfev=20000)
        pred = lognorm_pmf(x, *p)
        return 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum(), p
    except RuntimeError:
        return np.nan, None

def sub_edges(c, scale, name):
    e = set()
    for g, s_e, r_e in RES[c][scale, name]["subs"]:
        e.update(zip(s_e.tolist(), r_e.tolist()))
    return np.array(sorted(e), dtype=np.int64).reshape(-1, 2)

def fig4ac_table(c):
    past, rule = CONFIGS[c][:2]
    rows, curves = [], {}
    for scale in hdn.TIME_SCALES:
        for reading in ["f", "s"]:
            for measure in ["te", "nte"]:
                vals = []
                for r in recs:
                    net = NETS[past, scale, r["name"]]
                    if reading == "f":
                        v = net[measure][net["sig"][rule]]
                    else:
                        e = sub_edges(c, scale, r["name"])
                        v = net[measure][e[:, 0], e[:, 1]]
                    if (v > 0).sum() > 0:
                        vals.append(v[v > 0])
                pooled = np.concatenate(vals)
                edges = np.logspace(np.log10(pooled.min()), np.log10(pooled.max()), 101)
                centers = np.sqrt(edges[:-1] * edges[1:])
                pmfs = np.array([np.histogram(v, edges)[0] / len(v) for v in vals])
                r2_avg, p = fit_r2(centers, pmfs.mean(0))
                r2_each = np.array([fit_r2(centers, y)[0] for y in pmfs])
                p1, p99 = np.percentile(pooled, [1, 99])
                spread_each = np.array([np.percentile(v, 99) / np.percentile(v, 1) for v in vals])
                curves[scale, reading, measure] = (centers, pmfs, p)
                rows.append({"scale": scale, "reading": reading, "measure": measure, "recordings": len(vals), "n_values": len(pooled),
                             "R2_average": r2_avg, "R2_rec_median": np.nanmedian(r2_each), "R2_rec_min": np.nanmin(r2_each),
                             "p1": p1, "p99": p99, "p99/p1": p99 / p1, "p99/p1_rec_min": spread_each.min(),
                             "min": pooled.min(), "max": pooled.max(), "mu_log10": p[1] if p is not None else np.nan,
                             "sigma_log10": p[2] if p is not None else np.nan})
    return pd.DataFrame(rows), curves

def fig4ac_verdict(t, reading):
    sub = t[t.reading == reading]
    return "reproduced" if bool(((sub.R2_average >= 0.9) & (sub["p99/p1"] >= 100)).all()) else "not met"

fig4, curves = fig4ac_table("main")
print(fig4.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nCriterion (each reading): R2 >= 0.9 and p99/p1 >= 100 for both scales and both measures")
rows = []
for v, (c, form) in VARIANTS.items():
    if form != "claims":
        continue
    t = fig4 if c == "main" else fig4ac_table(c)[0]
    for reading in ["f", "s"]:
        sub = t[t.reading == reading]
        rows.append({"variant": v, "reading": reading, "R2_average": f"{sub.R2_average.min():.2f} to {sub.R2_average.max():.2f}",
                     "R2_rec_median": f"{sub.R2_rec_median.min():.2f} to {sub.R2_rec_median.max():.2f}",
                     "p99/p1": f"{sub['p99/p1'].min():.3g} to {sub['p99/p1'].max():.3g}", "criterion": fig4ac_verdict(t, reading)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, key in zip(axes.flat, curves):
    centers, pmfs, p = curves[key]
    ax.semilogx(centers, pmfs.mean(0), "k.", ms=3)
    if p is not None:
        ax.semilogx(centers, lognorm_pmf(centers, *p), "r-")
    ax.set_title(" ".join(key), fontsize=9)
plt.tight_layout(); plt.show()
print("Main analysis. Every bin of the 100 log-spaced bins is shown; values span the full range of each panel's x axis.")

**Result: partially reproduced, resting on the assumptions of the tie rule and bins t-2 and t-3.**
The significant TE values span several orders of magnitude: the 99th percentile is 353 to 1160
times the 1st, at both time scales, for raw and normalized TE and under both readings. The log-normal
of Eq 21 fits the distribution averaged over recordings with R2 of 0.935 and 0.957 for raw TE
(reading f) and 0.882 and 0.904 for normalized TE; under (s) 0.939 and 0.960, and 0.895 and 0.899.
Three of the eight fits fall just below the criterion's 0.9, all for normalized TE. With bins t-1 and
t-2 every fit reaches 0.90 to 0.98 and the criterion holds under both readings; with the strict
tie rule the averaged fits have R2 of 0.08 to 0.71.

## fig4bd-heavy-tailed-degree

> The total degree distributions are heavy-tailed, with many more high-degree neurons than in random
> networks with the same numbers of neurons and connections.

Random networks place the same number of directed connections uniformly among the same neurons:
under (f) 100 random networks per recording, under (s) one per sub-network (50 neurons, 75
connections). The threshold is the 99.9th percentile of the random total degrees of the recording
(f) or of all its random sub-networks (s).

In [ ]:
def random_degrees(n, e, rng):
    flat = rng.choice(n * (n - 1), e, replace=False)
    s = flat // (n - 1); r = flat % (n - 1); r = r + (r >= s)
    return np.bincount(s, minlength=n), np.bincount(r, minlength=n)

def fig4bd_table(c, rng):
    past, rule = CONFIGS[c][:2]
    deg, rows = {}, []
    for scale in hdn.TIME_SCALES:
        for reading in ["f", "s"]:
            above, n_real, max_real, max_rand, per_rec = 0, 0, 0, 0, []
            for r in recs:
                if reading == "f":
                    sig = NETS[past, scale, r["name"]]["sig"][rule]; n = sig.shape[0]; e = int(sig.sum())
                    ins, outs = [sig.sum(0)], [sig.sum(1)]
                    rt = np.concatenate([np.add(*random_degrees(n, e, rng)) for _ in range(100)])
                else:
                    ins, outs, rt = [], [], []
                    for g, s_e, r_e in RES[c][scale, r["name"]]["subs"]:
                        loc = {v: k for k, v in enumerate(g.tolist())}
                        s_l = np.array([loc[v] for v in s_e.tolist()]); r_l = np.array([loc[v] for v in r_e.tolist()])
                        ins.append(np.bincount(r_l, minlength=len(g))); outs.append(np.bincount(s_l, minlength=len(g)))
                        rt.append(np.add(*random_degrees(len(g), len(s_e), rng)))
                    if not rt:
                        continue
                    rt = np.concatenate(rt)
                ins, outs = np.concatenate(ins), np.concatenate(outs)
                tot = ins + outs
                thr = np.percentile(rt, 99.9)
                above += int((tot > thr).sum()); n_real += len(tot)
                max_real = max(max_real, tot.max()); max_rand = max(max_rand, rt.max())
                per_rec.append({"rec": r["name"], "in": ins, "out": outs, "tot": tot, "rand": rt, "thr": thr})
            deg[scale, reading] = per_rec
            rows.append({"scale": scale, "reading": reading, "recordings": len(per_rec), "neurons_or_slots": n_real,
                         "fraction_above_random_99.9": above / n_real, "ratio_to_0.001": above / n_real / 0.001,
                         "max_real_total": int(max_real), "max_random_total": int(max_rand),
                         "median_threshold": np.median([p["thr"] for p in per_rec])})
    return pd.DataFrame(rows), deg

def fig4bd_verdict(t, reading):
    sub = t[t.reading == reading]
    ok = ((sub["fraction_above_random_99.9"] >= 0.01) & (sub.max_real_total > sub.max_random_total)).all()
    return "reproduced" if bool(ok) else "not met"

rng = np.random.default_rng(41)
DEG = {}
fig4bd, DEG["main"] = fig4bd_table("main", rng)
print(fig4bd.round(4).to_string(index=False))
print("\nCriterion: fraction >= 0.01 and max real > max random, at both scales")
rows = []
for v, (c, form) in VARIANTS.items():
    if form != "claims":
        continue
    if c == "main":
        t = fig4bd
    else:
        t, DEG[c] = fig4bd_table(c, rng)
    for reading in ["f", "s"]:
        sub = t[t.reading == reading]
        rows.append({"variant": v, "reading": reading, "fraction_above": " / ".join(f"{x:.4f}" for x in sub["fraction_above_random_99.9"]),
                     "max_real": " / ".join(str(x) for x in sub.max_real_total), "max_random": " / ".join(str(x) for x in sub.max_random_total),
                     "criterion": fig4bd_verdict(t, reading)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, key in zip(axes, DEG["main"]):
    tot = np.concatenate([p["tot"] for p in DEG["main"][key]]); rt = np.concatenate([p["rand"] for p in DEG["main"][key]])
    bins = np.arange(0, max(tot.max(), rt.max()) + 2)
    ax.loglog(bins[:-1], np.histogram(tot, bins, density=True)[0], "k.", label="real total")
    ax.loglog(bins[:-1], np.histogram(rt, bins, density=True)[0], "r.", label="random total")
    ax.set_title(" ".join(key)); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Main analysis. Zero-degree neurons and empty bins are not shown on the log axes.")

**Result: reproduced.** In the full networks 10.2% (1.6 to 6.4 ms) and 14.3% (3.5 to 14 ms) of
neurons have a total degree above the 99.9th percentile of matched random networks, 102 and 143
times the 0.001 expected; in the sub-networks 7.1% and 6.6%. The largest real total degree is 373
and 517 (full) and 53 and 52 (sub-networks), against 84, 147, 14 and 13 in the random networks.
The criterion holds under every variant shown (strict rule, bins t-1 and t-2, random sub-network
connections).

## fig4bd-out-tail-longer

> The in-degree distribution has a shorter tail than the out-degree distribution, so there are more
> high out-degree neurons than high in-degree neurons.

In [ ]:
def tail_table(deg):
    rows = []
    for (scale, reading), per_rec in deg.items():
        ins = np.concatenate([p["in"] for p in per_rec]); outs = np.concatenate([p["out"] for p in per_rec])
        q_in = np.array([np.percentile(p["in"], 99) for p in per_rec]); q_out = np.array([np.percentile(p["out"], 99) for p in per_rec])
        w = stats.wilcoxon(q_out, q_in, alternative="greater") if np.any(q_out != q_in) else None
        rows.append({"scale": scale, "reading": reading, "p99_out": np.percentile(outs, 99), "p99_in": np.percentile(ins, 99),
                     "max_out": outs.max(), "max_in": ins.max(), "recordings_out_gt_in": int((q_out > q_in).sum()),
                     "recordings_out_lt_in": int((q_out < q_in).sum()), "wilcoxon_p_one_sided": w.pvalue if w else np.nan})
    return pd.DataFrame(rows)

def tail_verdict(t, reading):
    sub = t[t.reading == reading]
    return "reproduced" if bool(((sub.p99_out > sub.p99_in) & (sub.max_out > sub.max_in)).all()) else "not met"

tail = tail_table(DEG["main"])
print(tail.round(4).to_string(index=False))
print("\nCriterion: pooled p99 and max of out-degree above those of in-degree, at both scales")
rows = []
for v, (c, form) in VARIANTS.items():
    if form != "claims":
        continue
    t = tail_table(DEG[c])
    for reading in ["f", "s"]:
        sub = t[t.reading == reading]
        rows.append({"variant": v, "reading": reading, "p99_out/in": " / ".join(f"{a:.3g}-{b:.3g}" for a, b in zip(sub.p99_out, sub.p99_in)),
                     "max_out/in": " / ".join(f"{a}-{b}" for a, b in zip(sub.max_out, sub.max_in)),
                     "wilcoxon_p": " / ".join(f"{x:.2g}" for x in sub.wilcoxon_p_one_sided), "criterion": tail_verdict(t, reading)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced, resting on the tie rule.** Pooled over recordings, the 99th percentile of
out-degree exceeds that of in-degree at both time scales in the full networks (123 against 48, 209
against 98) and in the sub-networks (23 against 6, 26 against 6), and so does the largest
out-degree (322 against 86 and 395 against 180; 47 against 14 and 49 against 14). The per-recording
99th percentile of out-degree is higher in 25 and 25 of 25 recordings (full; one-sided Wilcoxon
p = 6.1e-06 and 3e-08) and in 23 and 25 (sub-networks). Both readings now meet the criterion. Under
the strict rule the full networks fail it at 3.5 to 14 ms (largest in-degree 480 against out-degree
449); every other variant meets it.

## fig5a-in-degree-uncorrelated, fig5b-out-degree-positive

> fig5a: The correlation between a receiver's in-degree and the synergy (computation) it performs is
> centered near zero across recordings, with a slight negative skew; computation is independent of
> in-degree.

> fig5b: Most recordings show a positive correlation between a transmitter's out-degree and the
> synergy computed by its receivers.

Synergy is normalized by H(I_F). Its redundancy term is the claims file's M-PID form (minimum over
sources within each state of I_F and I_P); Eq 8 of the version of record (minimum per state of
I_F, `hdn.pid_from_table`) is the variant "PID Eq 8". Under (s) each triple of each sub-network is
a data point, with degrees counted inside the sub-network, the 500 sub-networks pooled per
recording; under (f) the triples of the full network (sampled, see the table above) with
full-network degrees. Each transmitter of a triple is a separate data point for the out-degree.
Correlations: (p) Pearson, (r) Spearman.

In [ ]:
def triple_data(c, form, scale, name, reading):
    a = RES[c][scale, name]
    sd = a["syn"] if form == "claims" else a["syn8"]
    if reading == "s":
        t = a["sub_triples"]
        syn = np.array([sd[(x[1], x[2], x[3])] for x in t])
        return t[:, 4], syn, np.r_[t[:, 5], t[:, 6]], np.r_[syn, syn]
    t = a["full_triples"]
    syn = np.array([sd[tuple(x)] for x in t.tolist()])
    return a["indeg_f"][t[:, 0]], syn, np.r_[a["outdeg_f"][t[:, 1]], a["outdeg_f"][t[:, 2]]], np.r_[syn, syn]

def corr_table(c, form):
    rows = []
    for scale in hdn.TIME_SCALES:
        for r in recs:
            for reading in ["s", "f"]:
                indeg, syn_in, outdeg, syn_out = triple_data(c, form, scale, r["name"], reading)
                for kind, f in [("p", stats.pearsonr), ("r", stats.spearmanr)]:
                    for panel, x, y in [("in", indeg, syn_in), ("out", outdeg, syn_out)]:
                        ok = len(x) > 2 and np.ptp(x) > 0 and np.ptp(y) > 0
                        cc = f(x, y) if ok else None
                        rows.append({"scale": scale, "recording": r["name"], "reading": reading, "coef": kind, "panel": panel,
                                     "n": len(x), "r": cc[0] if ok else np.nan, "p": cc[1] if ok else np.nan})
    return pd.DataFrame(rows)

def skew_test(r):
    r = r.dropna(); pos, neg = int((r > 0).sum()), int((r < 0).sum()); n = pos + neg
    return {"n": n, "positive": pos, "negative": neg, "median_r": r.median(),
            "P(neg skew this large or larger)": stats.binom.cdf(pos, n, 0.5),
            "P(pos skew this large or larger)": stats.binom.sf(pos - 1, n, 0.5)}

def skew_table(corr):
    rows = []
    for panel in ["in", "out"]:
        for reading in ["s", "f"]:
            for kind in ["p", "r"]:
                for scale in ["both"] + list(hdn.TIME_SCALES):
                    sub = corr[(corr.panel == panel) & (corr.reading == reading) & (corr.coef == kind)]
                    if scale != "both":
                        sub = sub[sub.scale == scale]
                    rows.append({"panel": panel, "reading": reading, "coef": kind, "scale": scale, **skew_test(sub.r)})
    return pd.DataFrame(rows)

def fig5a_verdict(row):
    if row["P(pos skew this large or larger)"] < 0.05:
        return "not reproduced"
    return "partially reproduced" if row["P(neg skew this large or larger)"] < 0.05 else "reproduced"

def fig5b_verdict(row):
    if row.positive > row.n / 2:
        return "reproduced" if row["P(pos skew this large or larger)"] < 0.05 else "partially reproduced"
    return "not reproduced"

CORR, SKEW = {}, {}
for v, (c, form) in VARIANTS.items():
    CORR[v] = corr_table(c, form)
    SKEW[v] = skew_table(CORR[v])
print("Correlations that could not be computed (fewer than 3 points, or one degree or one synergy value), main:",
      int(CORR["main"].r.isna().sum()), "of", len(CORR["main"]))
print(SKEW["main"].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
rows = []
for v, sk in SKEW.items():
    for reading in ["s", "f"]:
        for kind in ["p", "r"]:
            a = sk[(sk.panel == "in") & (sk.reading == reading) & (sk.coef == kind) & (sk.scale == "both")].iloc[0]
            b = sk[(sk.panel == "out") & (sk.reading == reading) & (sk.coef == kind) & (sk.scale == "both")].iloc[0]
            rows.append({"variant": v, "reading": reading, "coef": kind,
                         "in: pos/neg": f"{a.positive}/{a.negative}", "in: median r": round(a.median_r, 3),
                         "in: P neg": f"{a['P(neg skew this large or larger)']:.2g}", "in: P pos": f"{a['P(pos skew this large or larger)']:.2g}",
                         "fig5a": fig5a_verdict(a),
                         "out: pos/neg": f"{b.positive}/{b.negative}", "out: median r": round(b.median_r, 3),
                         "out: P pos": f"{b['P(pos skew this large or larger)']:.2g}", "fig5b": fig5b_verdict(b)})
fig5_var = pd.DataFrame(rows)
print()
print(fig5_var.to_string(index=False))

In [ ]:
corr = CORR["main"]
print("Per-recording correlations, main, reading (s), Pearson, by time scale:")
piv = corr[(corr.reading == "s") & (corr.coef == "p")].pivot_table(index="recording", columns=["panel", "scale"], values="r")
print(piv.round(3).to_string())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, panel in zip(axes, ["in", "out"]):
    v = corr[(corr.reading == "s") & (corr.coef == "p") & (corr.panel == panel)].r.dropna()
    ax.hist(v, bins=np.linspace(-1, 1, 41)); ax.axvline(0, color="k")
    ax.set_title(f"{panel}-degree vs synergy, (s), Pearson; range {v.min():.2f} to {v.max():.2f}")
plt.tight_layout(); plt.show()

**Result: partially reproduced, resting on the assumptions of reading (s), Pearson correlations and
the tie rule.** Under (s, Pearson) 18 of 49 correlations between receiver in-degree and synergy are
positive and 31 negative, median r = -0.037, binomial probability of a negative skew this large
0.043: a slight negative skew, significant at 0.05, which the criterion counts as partial (1.6 to
6.4 ms alone 6 of 24 positive, p = 0.011; 3.5 to 14 ms 12 of 25, p = 0.5). Under (s, Spearman) 24
of 49 are positive (p = 0.5), which the criterion counts as reproduced. Under (f) 10 and 11 of 50
are positive (p = 1.2e-05 and 4.5e-05), a significant negative skew. With bins t-1 and t-2 both
(s) readings give no significant skew; with random sub-network connections and with the strict rule
the skew is significantly negative in seven of eight readings. The redundancy form of Eq 8 gives the
same counts.

**Result: reproduced, resting on the tie rule.** Correlations between transmitter out-degree and
synergy are positive in 42 of 49 recordings and time scales under (s, Pearson), median r = 0.118,
binomial p = 1.8e-07 (22 of 24 and 20 of 25 at each time scale); 44 of 49 under (s, Spearman),
p = 3.8e-09; 47 of 50 and 50 of 50 under (f). Every variant meets the criterion under every reading
except the strict rule with (f, Spearman), 26 of 50 positive (p = 0.44), which the criterion counts
as partial.

## fig5ab-correlations-beyond-null

> Nearly all the per-recording degree and synergy correlations lie beyond a null of randomized degree
> and computation pairings, whose correlations sit near zero.

Ten permutations of degree against synergy per correlation (400 null values for 40 correlations
in the paper; here 10 for each of the 50).

In [ ]:
def beyond_table(c, form, rng):
    rows = []
    for reading in ["s", "f"]:
        for kind, f in [("p", stats.pearsonr), ("r", stats.spearmanr)]:
            for panel in ["in", "out"]:
                real, null = [], []
                for scale in hdn.TIME_SCALES:
                    for r in recs:
                        indeg, syn_in, outdeg, syn_out = triple_data(c, form, scale, r["name"], reading)
                        x, y = (indeg, syn_in) if panel == "in" else (outdeg, syn_out)
                        if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
                            continue
                        real.append(f(x, y)[0])
                        null += [f(rng.permutation(x), y)[0] for _ in range(10)]
                real, null = np.abs(np.array(real)), np.abs(np.array(null))
                thr = np.percentile(null, 99)
                rows.append({"reading": reading, "coef": kind, "panel": panel, "n_corr": len(real), "n_null": len(null),
                             "null_|r|_median": np.median(null), "null_|r|_p99": thr, "fraction_beyond": (real > thr).mean()})
    return pd.DataFrame(rows)

rng = np.random.default_rng(52)
rows = []
for v, (c, form) in VARIANTS.items():
    b = beyond_table(c, form, rng)
    if v == "main":
        print(b.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
        print("\nCriterion: at least 90% beyond the null's 99th percentile in each panel")
    for (reading, kind), sub in b.groupby(["reading", "coef"], sort=False):
        rows.append({"variant": v, "reading": reading, "coef": kind,
                     "fraction_beyond in/out": " / ".join(f"{x:.2f}" for x in sub.fraction_beyond),
                     "null_p99 in/out": " / ".join(f"{x:.3f}" for x in sub["null_|r|_p99"]),
                     "criterion": "reproduced" if bool((sub.fraction_beyond >= 0.9).all()) else "not met"})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced, resting on the assumptions of reading (s), Pearson correlations and
the tie rule.** The permuted null correlations sit near zero (median absolute value 0.002 to 0.004
under (s), 0.010 to 0.015 under (f)). Under (s, Pearson) 98% of the out-degree correlations and 88%
of the in-degree correlations lie beyond the null's 99th percentile (0.016 and 0.031); the in-degree
panel misses the criterion's 90%. Under the other readings the in-degree panel has 63% to 80% and the
out-degree panel 86% to 100%. Only the strict rule under (s) meets the criterion in both panels
(92% and 90%; 94% and 94%).

## fig5cd-pooled-trends

> Pooled over all neuron groups and recordings, synergy shows no trend with receiver in-degree and
> rises with transmitter out-degree, at both time scales.

In [ ]:
def trend_table(c, form):
    rows = []
    for reading in ["s", "f"]:
        for scale in hdn.TIME_SCALES:
            parts = [triple_data(c, form, scale, r["name"], reading) for r in recs]
            for panel in ["in", "out"]:
                x = np.concatenate([p[0] if panel == "in" else p[2] for p in parts])
                y = np.concatenate([p[1] if panel == "in" else p[3] for p in parts])
                d = pd.DataFrame({"deg": x, "syn": y}).groupby("deg").syn.agg(["count", "median"])
                d = d[d["count"] >= 20]
                rho, p = stats.spearmanr(d.index, d["median"]) if len(d) > 2 else (np.nan, np.nan)
                rows.append({"reading": reading, "scale": scale, "panel": panel, "degrees_kept": len(d),
                             "degree_range": f"{d.index.min()} to {d.index.max()}", "groups": int(d["count"].sum()),
                             "median_at_lowest": d["median"].iloc[0], "median_at_highest": d["median"].iloc[-1],
                             "spearman_rho": rho, "p": p})
    return pd.DataFrame(rows)

def trend_verdict(t, reading):
    sub = t[t.reading == reading]
    o, i = sub[sub.panel == "out"], sub[sub.panel == "in"]
    ok = ((o.spearman_rho > 0) & (o.p < 0.05)).all() and not ((i.spearman_rho > 0) & (i.p < 0.05)).any()
    return "reproduced" if bool(ok) else "not met"

rows = []
for v, (c, form) in VARIANTS.items():
    t = trend_table(c, form)
    if v == "main":
        print(t.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
        print("\nCriterion: out-degree rho > 0 with p < 0.05, and in-degree not significantly positive, at both scales")
    for reading in ["s", "f"]:
        sub = t[t.reading == reading]
        o, i = sub[sub.panel == "out"], sub[sub.panel == "in"]
        rows.append({"variant": v, "reading": reading,
                     "out rho (p)": " / ".join(f"{a:.2f} ({b:.2g})" for a, b in zip(o.spearman_rho, o.p)),
                     "in rho (p)": " / ".join(f"{a:.2f} ({b:.2g})" for a, b in zip(i.spearman_rho, i.p)),
                     "criterion": trend_verdict(t, reading)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** With all triples pooled, the per-degree median synergy does not rise
with transmitter out-degree under (s) (Spearman rho = -0.26, p = 0.083, and -0.20, p = 0.17) and
rises at only one time scale under (f) (rho = 0.03, p = 0.71, at 1.6 to 6.4 ms; 0.16, p = 0.011,
at 3.5 to 14 ms). With receiver in-degree it falls under (s) (rho = -0.84 and -0.88, p = 0.0013 and
7.5e-05) and shows no trend under (f) (rho = -0.14 and -0.05). The in-degree part holds under both
readings; the out-degree part fails. No variant meets the criterion.

## fig6-information-gain-not-rising

> The information gained by adding inputs to a neuron, a bound on its highest-order synergy, stays
> constant or decreases with the number of inputs at both time scales; exponential fits to the
> medians give small negative exponents, one with a 95% range overlapping zero.

Per recording, time scale and reading, 30 receivers with at least one input are sampled: under (f)
neurons with their significant inputs in the full network, under (s) neurons of a sub-network with
their inputs inside it. For each n, all combinations of n inputs or 100 sampled ones, whichever is
fewer (the paper: every receiver, up to 100 combinations). The variant "sub random" is not run
here (its (f) gains equal the main analysis's).

In [ ]:
def expo(n, A, k):
    return A * np.exp(k * n)

def gain_table(c):
    rows = []
    for reading in ["f", "s"]:
        for scale in hdn.TIME_SCALES:
            for norm, col in [("u (normalized)", 1), ("v (bits)", 0)]:
                per_n = {}
                for n_in in range(1, hdn.MAX_INPUTS + 1):
                    per_n[n_in] = np.array([v[col] for (sc, nm), a in RES[c].items() if sc == scale
                                            for v in a["gains"][reading].get(n_in, {}).values()])
                for first in [2, 1]:
                    ns = np.array([n for n in range(first, hdn.MAX_INPUTS + 1) if len(per_n[n]) > 0])
                    med = np.array([np.median(per_n[n]) for n in ns])
                    try:
                        p, cov = curve_fit(expo, ns, med, p0=[med[0] * np.exp(0.1 * ns[0]), -0.1], maxfev=20000)
                        se = np.sqrt(cov[1, 1]); tq = stats.t.ppf(0.975, len(ns) - 2)
                        k, lo, hi = p[1], p[1] - tq * se, p[1] + tq * se
                    except (RuntimeError, ValueError):
                        k = lo = hi = np.nan
                    rows.append({"reading": reading, "scale": scale, "norm": norm, "n_from": first,
                                 "neurons_per_n": "/".join(str(len(per_n[n])) for n in range(first, hdn.MAX_INPUTS + 1)),
                                 "medians": " ".join(f"{m:.3g}" for m in med), "k": k, "k_low": lo, "k_high": hi})
    return pd.DataFrame(rows)

def gain_verdict(sub):
    if (sub.k_low > 0).any() or sub.k.isna().any():
        return "not reproduced" if (sub.k_low > 0).any() else "no fit"
    return "reproduced" if bool((sub.k < 0).all()) else "partially reproduced"

rows = []
for v, (c, form) in VARIANTS.items():
    if form != "claims" or not CONFIGS[c][3]:
        continue
    g = gain_table(c)
    if v == "main":
        print(g.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
        print()
    for (reading, norm, first), sub in g.groupby(["reading", "norm", "n_from"], sort=False):
        rows.append({"variant": v, "reading": reading, "norm": norm, "n_from": first,
                     "k (1.6 / 3.5)": " / ".join(f"{x:.3f}" for x in sub.k),
                     "95% interval": " / ".join(f"{a:.3f} to {b:.3f}" for a, b in zip(sub.k_low, sub.k_high)),
                     "criterion": gain_verdict(sub)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced, resting on the assumption that the gain is normalized by H(I_F), as the
paper's PID terms are.** For the normalized gain the fitted exponent k is negative at both time
scales under both network readings and both starting values of n, and no 95% interval lies above
zero: under (f), from n = 2, k = -0.071 (-0.089 to -0.053) and -0.114 (-0.123 to -0.104); from n = 1
the interval at 1.6 to 6.4 ms reaches -0.003; under (s) k is -0.074 to -0.249. For the gain in bits
under (f) k is positive at 1.6 to 6.4 ms (0.008, interval -0.004 to 0.020, from n = 2), which the
criterion counts as partial; under (s) in bits it is reproduced from n = 2 and partial from n = 1.
With bins t-1 and t-2 the gain in bits under (f) rises at 1.6 to 6.4 ms with an interval above zero
(0.047, 0.036 to 0.059), which would be not reproduced; the normalized gain is reproduced under every
variant. The statement's detail that one interval overlaps zero holds only for the gain in bits.
Between 350 and 750 receivers contribute per n under (f), and 12 to 750 under (s), from 30 sampled
receivers per recording and time scale.

## table1-pid-examples

> On the example distributions of Table 1, the PID multivariate TE finds synergy only in the
> synergistic and the synergistic and redundant examples; mutual information misses the hidden self
> interaction, and interaction information misses simultaneous synergy and redundancy and unique
> information.

The joint distributions are the paper's Table 1 (rows I_F, I_P, J_P, K_P from 0000 to 1111), typed
in as printed; each column is renormalized to sum to 1. The PID uses the claims file's M-PID form
of the redundancy; the Eq 8 form is shown as a variant.

In [ ]:
TABLE1 = {  # p(I_F, I_P, J_P, K_P), rows 0000 ... 1111, as printed in Table 1
    "No Interaction": ["1/16"] * 16,
    "Self Interaction": ["1/8", "1/8", "1/8", "1/8", "0", "0", "0", "0", "0", "0", "0", "0", "1/8", "1/8", "1/8", "1/8"],
    "Hidden Self Interaction": ["1/4", "1/4", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "1/4", "1/4"],
    "Single Interaction": ["1/8", "1/8", "0", "0", "1/8", "1/8", "0", "0", "0", "0", "1/8", "1/8", "0", "0", "1/8", "1/8"],
    "Redundant Interaction": ["1/4", "0", "0", "0", "1/4", "0", "0", "0", "0", "0", "0", "1/4", "0", "0", "0", "1/4"],
    "Single and Redundant Interaction": ["0.1964", "0.0536", "0", "0", "0.1964", "0.0536", "0", "0", "0", "0", "0.0536", "0.1964", "0", "0", "0.0536", "0.1964"],
    "Synergistic Interaction": ["1/8", "0", "0", "1/8", "1/8", "0", "0", "1/8", "0", "1/8", "1/8", "0", "0", "1/8", "1/8", "0"],
    "Synergistic and Redundant Interaction": ["0.0548", "0.0548", "0.0548", "0", "0.0548", "0.0548", "0.0548", "0", "0", "0", "0", "0.3357", "0", "0", "0", "0.3357"],
}

def mi(pxy):
    px, py = pxy.sum(1, keepdims=True), pxy.sum(0, keepdims=True)
    m = pxy > 0
    return float((pxy[m] * np.log2(pxy[m] / (px @ py)[m])).sum())

rows = []
for name, col in TABLE1.items():
    p = np.array([eval(v) for v in col], float); p = p / p.sum()
    n = np.zeros((2, 2, 4))
    pf = np.zeros((2, 2, 2))   # p(I_F, J_P, K_P)
    for row in range(16):
        f, q, j, k = (row >> 3) & 1, (row >> 2) & 1, (row >> 1) & 1, row & 1
        n[f, q, j + 2 * k] = p[row] * 1e6
        pf[f, j, k] += p[row]
    syn, red, uJ, uK, teJ, teK, teJK, H, syn8, red8 = hdn.pid_from_table(n)
    mi_j = mi(pf.sum(2))                                  # MI(J_P; I_F)
    mi_jk = mi(pf.sum(0))                                 # MI(J_P; K_P)
    mi_jk_f = sum(pf[f].sum() * mi(pf[f] / pf[f].sum()) for f in range(2) if pf[f].sum() > 0)  # MI(J_P; K_P | I_F)
    for form, (s_, r_) in [("claims", (syn, red)), ("eq8", (syn8, red8))]:
        rows.append({"form": form, "example": name, "TE(J)": teJ, "TE(K)": teK, "TE(JK)": teJK, "redundancy": r_,
                     "unique J": teJ - r_, "unique K": teK - r_, "synergy": s_, "MI(J_P;I_F)": mi_j, "II": mi_jk_f - mi_jk})
t1_all = pd.DataFrame(rows)
tol = 0.001
for form in ["claims", "eq8"]:
    t1 = t1_all[t1_all.form == form].drop(columns="form").set_index("example")
    print(f"=== redundancy form: {form} ===")
    print(t1.round(4).to_string())
    hs, srd, snr = t1.loc["Hidden Self Interaction"], t1.loc["Synergistic and Redundant Interaction"], t1.loc["Single and Redundant Interaction"]
    checks = {
        "synergy > 0.001 exactly in the two synergistic examples":
            set(t1.index[t1.synergy > tol]) == {"Synergistic Interaction", "Synergistic and Redundant Interaction"},
        "hidden self: MI(J_P;I_F) > 0 while TE and every PID term are 0":
            hs["MI(J_P;I_F)"] > tol and all(abs(hs[c]) <= tol for c in ["TE(J)", "TE(K)", "TE(JK)", "redundancy", "unique J", "unique K", "synergy"]),
        "synergistic and redundant: II = 0 while synergy and redundancy are not":
            abs(srd["II"]) <= tol and srd.synergy > tol and srd.redundancy > tol,
        "single and redundant: unique J > 0 and |II| = redundancy":
            snr["unique J"] > tol and abs(abs(snr["II"]) - snr.redundancy) <= tol,
    }
    for k, v in checks.items():
        print(f"  {k}: {v}")

**Result: reproduced.** On the distributions of Table 1, synergy is above 0.001 bits only in the
synergistic (1.000 bit) and the synergistic and redundant (0.457 bits) examples. In the hidden self
interaction MI(J_P; I_F) is 1 bit while TE and every PID term are 0. In the synergistic and redundant
example the interaction information is 0.0001 bits while synergy and redundancy are 0.457 bits each.
In the single and redundant example the unique information of J is 0.750 bits and the interaction
information is -0.250 bits, the size of the redundancy (0.250 bits). The claims file's redundancy
form and that of Eq 8 give the same values on every example.

## Feedforward model: fig7b, fig7c, fig7d, figs1

> fig7c: In a two-layer feedforward model, degree-modified and firing-rate-modified Hebbian rewiring
> rules qualitatively reproduce the degree and synergy correlations of the real data, the
> degree-modified rule more closely, while a pure Hebbian rule and random wiring do not.

> fig7b: The pure Hebbian rule pools connections among the input neurons most correlated with the
> network-wide signal, the firing-rate-modified rule spreads them to less correlated input neurons,
> and the degree-modified rule spreads them almost uniformly across the input layer.

> fig7d: The pure Hebbian rule gives the highest mutual information between connected neurons, but
> conditioned on the network-wide binary signal the degree-modified rule gives the highest; it alone
> raises neuron-to-neuron communication while also reinforcing network-wide correlation.

> figs1: With 40 neurons per layer the model gives the same qualitative results, the degree-modified
> rule again raising neuron-to-neuron communication, though the Hebbian and firing-rate-modified
> rules give larger conditioned mutual information than in the small model.

The model follows the Methods and Eqs 13 to 24 of the version of record (`hdn.py`), whose displayed
equations read, as transcribed from its JATS XML:

- Eq 13, p(I) = 1 / (1 + e^(-alpha I + beta)); Eq 14, p(I = 0) = 0.01; Eq 15, p(I = I_con N_Neurons) = 0.5.
- Eqs 16 and 17, p(b(t) = 0, i) = 0.5 - 0.4 (i - 1)/(N_Neurons - 1) and p(b(t) = 1, i) = 0.5 + 0.4 (i - 1)/(N_Neurons - 1).
- Eqs 18 and 19, the same for output neuron o with 0.2 in place of 0.4.
- Eq 20, S(i, o) = A(i, o) + a1 DegIn(o) + a2 DegOut(i) + a3 FR(o) + a4 FR(i).
- Eq 22, S_Deg(i, o) = A(i, o) + 0.05 DegIn(o) - 1.75 DegOut(i); Eq 23, S_FR(i, o) = A(i, o) + 0.35 FR(o) + 3.1 FR(i);
  Eq 24, S_Deg,40(i, o) = A(i, o) + 0.16 DegIn(o) - 0.3 DegOut(i).

The PID equations read, for the record: Eq 8, Redundancy = sum over i_F of p(i_F) min over
R in {J_P, K_P} of [I_spec(I_F = i_F; R, I_P) - I_spec(I_F = i_F; I_P)], with Eq 9,
I_spec(I_F = i_F; R, I_P) = sum over r, i_P of p(r, i_P | i_F) log [p(r, i_P, i_F) / (p(r, i_P) p(i_F))],
and Eq 10 the same without r.

Synergy and MI are computed from 2000 fresh runs per model. The real data's reference is the mean
and standard deviation of the 50 per-recording correlations (25 recordings, two time scales) of
the main analysis, reading (s), Pearson; the other readings and variants are shown. Where all
output neurons of a model have the same in-degree, its in-degree correlation is undefined; the
model means are given over the defined models and, as a check, with the undefined ones set to 0.

In [ ]:
MODEL_EVAL_RUNS = 2000
models = {}
t_all = time.time()
for N, readings in [(20, ["kept"]), (40, ["recomputed", "kept"])]:
    for ab in readings:
        alpha, beta = hdn.sigmoid_params(N if ab == "recomputed" else 20)
        for k, rule in enumerate(hdn.RULES[N]):
            rng = np.random.default_rng(700 + 10 * N + k + (5 if ab == "kept" and N == 40 else 0))
            conn = hdn.build_models(N, rule, rng, alpha, beta)
            models[N, ab, rule] = (conn, pd.DataFrame(hdn.model_correlations(conn, rng, alpha, beta, MODEL_EVAL_RUNS)))
print(f"models built and evaluated in {time.time() - t_all:.0f} s")
print("alpha, beta for N = 20:", [round(v, 4) for v in hdn.sigmoid_params(20)], "| recomputed for N = 40:", [round(v, 4) for v in hdn.sigmoid_params(40)])
rows = []
for (N, ab, rule), (conn, d) in models.items():
    rows.append({"N": N, "alpha_beta": ab, "rule": rule, "models": len(d), "connections": int(conn[0].sum()),
                 **{c: d[c].mean() for c in ["in_p", "out_p", "in_r", "out_r", "mi", "mi_given_b", "upper_half_fraction"]},
                 "in_p_sd": d.in_p.std(), "out_p_sd": d.out_p.std(), "nan_in": int(d.in_p.isna().sum()), "nan_out": int(d.out_p.isna().sum())})
mod = pd.DataFrame(rows)
print(mod.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
def real_ref(v, reading, kind):
    sub = CORR[v][(CORR[v].reading == reading) & (CORR[v].coef == kind)]
    i, o = sub[sub.panel == "in"].r.dropna(), sub[sub.panel == "out"].r.dropna()
    return i.mean(), i.std(), o.mean(), o.std()

def match_table(N, ab, v, reading, kind, nan_zero=False):
    mi_, si, mo, so = real_ref(v, reading, kind)
    rows = []
    for rule in hdn.RULES[N]:
        d = models[N, ab, rule][1]
        xi, xo = d["in_" + kind], d["out_" + kind]
        if nan_zero:
            xi, xo = xi.fillna(0), xo.fillna(0)
        a, b = xi.mean(), xo.mean()
        rows.append({"rule": rule, "mean_in": a, "mean_out": b, "match": abs(a - mi_) <= si and abs(b - mo) <= so,
                     "distance": math.hypot(a - mi_, b - mo)})
    return pd.DataFrame(rows), (mi_, si, mo, so)

def fig7c_ok(t):
    m = t.set_index("rule")
    return bool(m.match["degree"] and m.match["firing_rate"] and not m.match["hebbian"] and not m.match["random"]
                and m.distance["degree"] < m.distance["firing_rate"])

def fig7c_label(t):
    m = t.set_index("rule")
    if fig7c_ok(t):
        return "reproduced"
    qual = bool(m.match["degree"] and m.match["firing_rate"] and not m.match["hebbian"] and not m.match["random"])
    return "match, but firing-rate rule closer" if qual else "not met"

for reading in ["s", "f"]:
    for kind in ["p", "r"]:
        t, ref = match_table(20, "kept", "main", reading, kind)
        print(f"main, reading ({reading}, {kind}): real in {ref[0]:.3f} +- {ref[1]:.3f}, out {ref[2]:.3f} +- {ref[3]:.3f}")
        print(t.round(4).to_string(index=False))
        print("  fig7c criterion:", fig7c_ok(t), "| with undefined in-degree correlations set to 0:",
              fig7c_label(match_table(20, "kept", "main", reading, kind, nan_zero=True)[0]))
rows = []
for v in VARIANTS:
    for reading in ["s", "f"]:
        for kind in ["p", "r"]:
            t, ref = match_table(20, "kept", v, reading, kind)
            m = t.set_index("rule")
            rows.append({"variant": v, "reading": reading, "coef": kind, "real in": f"{ref[0]:.3f} +- {ref[1]:.3f}",
                         "real out": f"{ref[2]:.3f} +- {ref[3]:.3f}", "matching rules": ",".join(m.index[m.match]),
                         "dist degree": round(m.distance["degree"], 3), "dist firing_rate": round(m.distance["firing_rate"], 3),
                         "fig7c": fig7c_label(t)})
print(pd.DataFrame(rows).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 4))
for rule in hdn.RULES[20]:
    d = models[20, "kept", rule][1]
    ax.plot(d.in_p, d.out_p, ".", alpha=0.3, label=rule)
mi_, si, mo, so = real_ref("main", "s", "p")
ax.errorbar(mi_, mo, xerr=si, yerr=so, color="k", label="real (s, p)")
ax.set_xlabel("in-degree vs synergy r"); ax.set_ylabel("out-degree vs synergy r"); ax.legend(fontsize=7)
plt.show()
d_all = pd.concat([models[20, "kept", r][1] for r in hdn.RULES[20]])
print(f"model correlations range: in {d_all.in_p.min():.3f} to {d_all.in_p.max():.3f}, out {d_all.out_p.min():.3f} to {d_all.out_p.max():.3f}; all within the axes")

**Result: partially reproduced, resting on the assumptions of reading (s), Pearson correlations and
the tie rule.** Against the real mean and standard deviation (in-degree -0.016 +- 0.117, out-degree
0.127 +- 0.120), the degree-modified models (0.017, 0.114) match and are the nearest (distance
0.036), and the Hebbian (0.321, 0.022) and random (0.128, -0.009) models do not match, as the paper
says. The firing-rate-modified models (-0.061, 0.280) do not match: their out-degree correlation is
beyond one standard deviation. Under (s, Spearman) the whole criterion holds (degree distance 0.019
against 0.138); under (f) only the firing-rate rule matches. Counting the 56 degree-modified models
with undefined in-degree correlations as 0 changes no verdict.

In [ ]:
def fig7b_table(N, ab):
    t = pd.DataFrame([{"rule": r, "upper_half_fraction": models[N, ab, r][1].upper_half_fraction.mean(),
                       "sd": models[N, ab, r][1].upper_half_fraction.std()} for r in hdn.RULES[N]])
    m = t.set_index("rule").upper_half_fraction
    ok = bool(m["hebbian"] > m["firing_rate"] > m["degree"] and abs(m["degree"] - 0.5) <= 0.1)
    return t, ok

t, ok = fig7b_table(20, "kept")
print(t.round(4).to_string(index=False)); print("fig7b criterion:", ok)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, rule in zip(axes, hdn.RULES[20]):
    im = ax.imshow(models[20, "kept", rule][0].mean(0), vmin=0, vmax=1, cmap="viridis")
    ax.set_title(rule); ax.set_xlabel("output o"); ax.set_ylabel("input i")
plt.colorbar(im, ax=axes); plt.show()

**Result: reproduced.** The fraction of connections leaving the upper half of the input layer is
0.988 for the Hebbian rule, 0.792 for the firing-rate-modified rule and 0.505 for the degree-modified
rule (random wiring 0.502), in the order the criterion requires, with the degree-modified rule
within 0.5 +- 0.1.

In [ ]:
def fig7d_tests(N, ab):
    rules = list(hdn.RULES[N])
    rows = []
    for col in ["mi", "mi_given_b"]:
        for a, b in itertools.combinations(rules, 2):
            x, y = models[N, ab, a][1][col], models[N, ab, b][1][col]
            rows.append({"measure": col, "a": a, "b": b, "mean_a": x.mean(), "mean_b": y.mean(),
                         "p": stats.mannwhitneyu(x, y, alternative="two-sided").pvalue})
    t = pd.DataFrame(rows)
    t["p_fdr"] = stats.false_discovery_control(t.p, method="bh")
    return t

def above(t, N, ab, hi, lo):
    """hi's mean conditioned MI above lo's, at FDR-corrected p < 0.001."""
    c = t[(t.measure == "mi_given_b") & (((t.a == hi) & (t.b == lo)) | ((t.a == lo) & (t.b == hi)))]
    return bool(models[N, ab, hi][1].mi_given_b.mean() > models[N, ab, lo][1].mi_given_b.mean() and c.p_fdr.iloc[0] < 0.001)

def fig7d_ok(N, ab, t):
    means = {r: models[N, ab, r][1][["mi", "mi_given_b"]].mean() for r in hdn.RULES[N]}
    top_u = max(means, key=lambda r: means[r]["mi"]); top_c = max(means, key=lambda r: means[r]["mi_given_b"])
    deg_beats = all(above(t, N, ab, "degree", r) for r in hdn.RULES[N] if r != "degree")
    not_above_random = all(not above(t, N, ab, r, "random") for r in ["hebbian", "firing_rate"])
    return top_u, top_c, deg_beats, not_above_random

t = fig7d_tests(20, "kept")
print(t.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
top_u, top_c, deg_beats, nar = fig7d_ok(20, "kept", t)
print(f"highest unconditioned MI: {top_u}; highest conditioned MI: {top_c}; degree above every other rule at FDR p < 0.001: {deg_beats}; "
      f"Hebbian and firing-rate not significantly above random: {nar}")
print("fig7d criterion:", top_u == "hebbian" and top_c == "degree" and deg_beats and nar)
# first-order bias of the plug-in MI of two binary variables, (2-1)(2-1) / (2 n ln 2) bits
bias_u = 1 / (2 * MODEL_EVAL_RUNS * np.log(2))
bias_c = 1 / (2 * (MODEL_EVAL_RUNS / 2) * np.log(2))
print(f"Plug-in bias, first order: {bias_u:.2e} bits unconditioned, {bias_c:.2e} bits conditioned on b (two halves of {MODEL_EVAL_RUNS // 2} runs);"
      " the same for every rule, since every model uses the same number of runs and binary states.")
print("Conditioned MI minus that bias:", {r: f"{models[20, 'kept', r][1].mi_given_b.mean() - bias_c:.2e}" for r in hdn.RULES[20]})

**Result: not reproduced.** The Hebbian rule gives the highest unconditioned MI between connected
neurons (0.090 bits, against 0.071, 0.030 and 0.023), as the paper says. Conditioned on the binary
signal, the random model gives the highest MI (0.00230 bits) and the degree-modified rule the lowest
(0.00153; Hebbian 0.00168, firing-rate 0.00166). The degree-modified rule differs from the random,
Hebbian and firing-rate rules at FDR p < 0.001 but lies below each, so it is above none. The
first-order plug-in bias of the conditioned MI, 7.2e-04 bits, is the same for every rule; with it
subtracted the order is unchanged.

In [ ]:
rows = []
for ab in ["recomputed", "kept"]:
    print(f"=== N = 40, alpha and beta {ab} ===")
    tb, ok_b = fig7b_table(40, ab)
    print(tb.round(4).to_string(index=False), "| connectivity ordering:", ok_b)
    td = fig7d_tests(40, ab)
    print(td.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    deg_above = above(td, 40, ab, "degree", "random")
    larger = {r: (models[40, ab, r][1].mi_given_b.mean(), models[20, "kept", r][1].mi_given_b.mean()) for r in ["hebbian", "firing_rate"]}
    print("degree conditioned MI above random (FDR p < 0.001):", deg_above)
    print("conditioned MI, N = 40 against N = 20:", {r: (round(a, 5), round(b, 5)) for r, (a, b) in larger.items()})
    for v in VARIANTS:
        for reading, kind in [("s", "p"), ("s", "r"), ("f", "p"), ("f", "r")]:
            t, ref = match_table(40, ab, v, reading, kind)
            m = t.set_index("rule")
            ok_c = bool(m.match["degree"] and m.match["firing_rate"] and not m.match["hebbian"])
            rows.append({"alpha_beta": ab, "variant": v, "reading": reading, "coef": kind,
                         "mean_in": " ".join(f"{x:.3f}" for x in t.mean_in), "mean_out": " ".join(f"{x:.3f}" for x in t.mean_out),
                         "correlation match": ok_c, "ordering": ok_b, "degree MI above random": deg_above,
                         "Hebbian and FR larger than N=20": all(a > b for a, b in larger.values()),
                         "figs1 criterion": ok_c and ok_b and deg_above and all(a > b for a, b in larger.values())})
print("\nRule order in mean_in and mean_out:", list(hdn.RULES[40]))
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** With 40 neurons per layer the connectivity ordering holds (Hebbian
0.991, firing-rate 0.769, degree 0.502 with alpha and beta recomputed; 0.999, 0.798, 0.501 with them
kept). The degree-modified rule's conditioned MI is below the random model's under both readings
(0.00094 against 0.00112; 0.00100 against 0.00206), and the Hebbian and firing-rate rules'
conditioned MI is smaller than in the 20-neuron model (0.00096 and 0.00097, or 0.00147 and 0.00146,
against 0.00168 and 0.00166), not larger. The correlation match fails under every reading of the main
analysis. The criterion fails under every reading, variant and choice of alpha and beta.

## Summary

| Claim | Outcome |
|---|---|
| data-recordings | reproduced |
| fig4ac-te-lognormal | partially reproduced (assumes the tie rule and bins t-2, t-3) |
| fig4bd-heavy-tailed-degree | reproduced |
| fig4bd-out-tail-longer | reproduced (assumes the tie rule) |
| fig5a-in-degree-uncorrelated | partially reproduced (assumes sub-networks, Pearson, the tie rule) |
| fig5b-out-degree-positive | reproduced (assumes the tie rule) |
| fig5ab-correlations-beyond-null | partially reproduced (assumes sub-networks, Pearson, the tie rule) |
| fig5cd-pooled-trends | not reproduced |
| fig6-information-gain-not-rising | reproduced (assumes the normalized gain) |
| table1-pid-examples | reproduced |
| fig7b-connectivity-patterns | reproduced |
| fig7c-degree-modified-rule-matches | partially reproduced (assumes sub-networks, Pearson, the tie rule) |
| fig7d-degree-rule-highest-conditioned-mi | not reproduced |
| figs1-larger-model | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper reused TE networks computed for [44]; they are rebuilt here from the deposited spike times with the bins, past states, jitter window and threshold of [44], so they need not be the same networks. A surrogate that ties with the real TE counts as exceeding it; [44]'s text says "larger", which, taken literally, calls a pair significant whenever the jitter cannot change its counts, and gives a false-positive rate of 10.8% on independent trains here. Which two bins form the past state is read from the delay windows both papers state (t-2 and t-3); bins t-1 and t-2, from [44]'s text, are shown as a variant. For pairs whose jitter moves more than 50 relevant spikes the surrogate counts are drawn from a normal distribution with their exact moments; on two recordings this shares 83% to 89% of the connections with the full jitter. Sub-networks with fewer than 75 connections are redrawn, and four recordings give fewer than 500 at 1.6 to 6.4 ms. The redundancy uses the claims file's form, which differs from Eq 8 of the version of record; Eq 8 is shown and changes no outcome. The full-network triples are sampled (3000 per recording and time scale), and the information gain uses 30 receivers per recording, time scale and reading (the paper: all receivers). The variants use 100 sub-networks per recording and 20 combinations per n. The model's synergy is the PID of two input states about the output state in the same run, without the receiver's past, which the model does not have; MI and synergy use 2000 fresh runs per model.

What was found about the data set. The deposit matches the paper's description: 25 recordings, 7735 neurons, all with at least 100 spikes, a mean rate of 2.10 Hz, spike times on the 20 kHz grid. One recording (DataSet19) lasts 48 min, not 60. The first ten recordings have no spikes in their first 8 s. The networks of [44] that the paper analysed are not deposited, so the networks have to be rebuilt.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")